# Day 60 — LLM Fine-Tuning Theory
### When to fine-tune, LoRA/QLoRA explained, PEFT library, environment setup for Day 61

## 1. Setup

In [1]:
import pandas as pd

pd.set_option('display.max_colwidth', None)
print("Ready")

Ready


## 2. Fine-Tuning vs Prompt Engineering vs RAG — Decision Framework

In [2]:
decision_data = [
    {"approach": "Prompt Engineering", "cost": "$ (cheapest)", "setup_time": "Minutes",
     "data_needed": "0 examples (or few-shot)", "best_for": "General tasks, quick iteration, no domain shift needed"},
    {"approach": "RAG", "cost": "$$ (low-medium)", "setup_time": "Hours-Days",
     "data_needed": "Document corpus", "best_for": "Knowledge that changes often, need citations/grounding"},
    {"approach": "Fine-Tuning (LoRA/QLoRA)", "cost": "$$$ (medium)", "setup_time": "Days",
     "data_needed": "100s-1000s of examples", "best_for": "Consistent style/format, domain-specific behavior, latency-critical"},
    {"approach": "Full Fine-Tuning", "cost": "$$$$ (expensive)", "setup_time": "Weeks",
     "data_needed": "10,000s+ examples", "best_for": "Fundamentally new capability, massive scale deployment"},
]

df = pd.DataFrame(decision_data)
df

,approach,cost,setup_time,data_needed,best_for
0,Prompt Engineering,$ (cheapest),Minutes,0 examples (or few-shot),"General tasks, quick iteration, no domain shift needed"
1,RAG,$$ (low-medium),Hours-Days,Document corpus,"Knowledge that changes often, need citations/grounding"
2,Fine-Tuning (LoRA/QLoRA),$$$ (medium),Days,100s-1000s of examples,"Consistent style/format, domain-specific behavior, latency-critical"
3,Full Fine-Tuning,$$$$ (expensive),Weeks,"10,000s+ examples","Fundamentally new capability, massive scale deployment"


## 3. The Golden Rule — Try Cheaper Options First

In [3]:
golden_rule = """
THE GOLDEN RULE OF LLM CUSTOMIZATION:

Always try the cheapest option first, and only escalate when it genuinely fails.

1. Start with PROMPT ENGINEERING
   -> Can better instructions, examples, or system prompts solve this?
   -> 80% of "I need to fine-tune" problems are actually solved here

2. If knowledge is the gap, try RAG
   -> Is the model failing because it doesn't KNOW something?
   -> RAG fixes knowledge gaps without touching model weights

3. Only fine-tune if you need:
   -> A specific STYLE or FORMAT the model can't reliably produce via prompting
   -> Behaviour that's hard to specify in words (tone, brand voice)
   -> Lower latency (smaller fine-tuned model vs large model + huge prompt)
   -> To run fully offline / on-device

4. Full fine-tuning is rarely the right choice for individuals or small teams
   -> Requires massive compute, massive data, massive cost
   -> LoRA/QLoRA achieve 90%+ of the benefit at 1% of the cost
"""

print(golden_rule)


THE GOLDEN RULE OF LLM CUSTOMIZATION:

Always try the cheapest option first, and only escalate when it genuinely fails.

1. Start with PROMPT ENGINEERING
   -> Can better instructions, examples, or system prompts solve this?
   -> 80% of "I need to fine-tune" problems are actually solved here

2. If knowledge is the gap, try RAG
   -> Is the model failing because it doesn't KNOW something?
   -> RAG fixes knowledge gaps without touching model weights

3. Only fine-tune if you need:
   -> A specific STYLE or FORMAT the model can't reliably produce via prompting
   -> Behaviour that's hard to specify in words (tone, brand voice)
   -> Lower latency (smaller fine-tuned model vs large model + huge prompt)
   -> To run fully offline / on-device

4. Full fine-tuning is rarely the right choice for individuals or small teams
   -> Requires massive compute, massive data, massive cost
   -> LoRA/QLoRA achieve 90%+ of the benefit at 1% of the cost



## 4. How LoRA Works — The Math Made Simple

In [4]:
import numpy as np

# demonstrate the core LoRA insight with a tiny example
print("=== Standard Fine-Tuning (full weight update) ===")
d = 8  # pretend this is a tiny slice of a real weight matrix (real ones are 1000s x 1000s)
W = np.random.randn(d, d)
print(f"Original weight matrix W: {d}x{d} = {d*d} parameters to update")
print()

print("=== LoRA's Insight ===")
print("Instead of updating ALL d x d parameters, LoRA decomposes the UPDATE")
print("into two small matrices A and B, where r (rank) is much smaller than d:")
print()

r = 2  # the "rank" — much smaller than d
A = np.random.randn(d, r) * 0.01
B = np.zeros((r, d))  # B starts at zero so the update starts as zero

lora_params = (d * r) + (r * d)
full_params = d * d

print(f"A matrix: {d}x{r} = {d*r} parameters")
print(f"B matrix: {r}x{d} = {r*d} parameters")
print(f"Total LoRA parameters: {lora_params}")
print(f"Full fine-tuning parameters: {full_params}")
print(f"Reduction: {round((1 - lora_params/full_params) * 100, 1)}% fewer parameters to train")
print()

print("During training, only A and B are updated.")
print("The original W stays completely frozen.")
print("At inference: output = W*x + (B*A)*x  -- the LoRA update is ADDED to the frozen weights")

=== Standard Fine-Tuning (full weight update) ===
Original weight matrix W: 8x8 = 64 parameters to update

=== LoRA's Insight ===
Instead of updating ALL d x d parameters, LoRA decomposes the UPDATE
into two small matrices A and B, where r (rank) is much smaller than d:

A matrix: 8x2 = 16 parameters
B matrix: 2x8 = 16 parameters
Total LoRA parameters: 32
Full fine-tuning parameters: 64
Reduction: 50.0% fewer parameters to train

During training, only A and B are updated.
The original W stays completely frozen.
At inference: output = W*x + (B*A)*x  -- the LoRA update is ADDED to the frozen weights


## 5. Real-World LoRA Numbers — Why It Matters at Scale

In [5]:
def lora_savings(d, r):
    full_params = d * d
    lora_params = (d * r) + (r * d)
    reduction = round((1 - lora_params / full_params) * 100, 2)
    return full_params, lora_params, reduction

scenarios = [
    ("Toy example", 8, 2),
    ("Small LLM layer", 1024, 8),
    ("Real LLM layer (Llama-style)", 4096, 16),
    ("Large LLM layer", 8192, 32),
]

print(f"{'Scenario':<32} {'d':>6} {'r':>4} {'Full params':>14} {'LoRA params':>14} {'Reduction':>10}")
print("-" * 85)
for name, d, r in scenarios:
    full, lora, reduction = lora_savings(d, r)
    print(f"{name:<32} {d:>6} {r:>4} {full:>14,} {lora:>14,} {reduction:>9}%")

Scenario                              d    r    Full params    LoRA params  Reduction
-------------------------------------------------------------------------------------
Toy example                           8    2             64             32      50.0%
Small LLM layer                    1024    8      1,048,576         16,384     98.44%
Real LLM layer (Llama-style)       4096   16     16,777,216        131,072     99.22%
Large LLM layer                    8192   32     67,108,864        524,288     99.22%


## 6. QLoRA — Quantization + LoRA

In [6]:
qlora_explanation = """
QLoRA = Quantized LoRA

LoRA already reduces TRAINABLE parameters dramatically.
QLoRA additionally reduces MEMORY usage of the FROZEN base model.

The problem QLoRA solves:
- A 7B parameter model in standard 16-bit precision needs ~14GB of GPU memory
  just to LOAD the model -- before any training even happens
- Most free/affordable GPUs (Colab free tier, consumer GPUs) have 8-16GB VRAM
- This makes fine-tuning even with LoRA inaccessible on modest hardware

QLoRA's fix:
1. Load the frozen base model in 4-bit precision instead of 16-bit
   -> Cuts memory for the frozen weights by ~4x
2. Apply LoRA adapters (in higher precision) on top of the quantized base
   -> Only the small LoRA matrices need higher precision for stable training
3. Use "double quantization" and paged optimizers for further memory savings

Result: a 7B model that needed ~14GB now fits in ~5-6GB
-> Makes fine-tuning possible on free Google Colab GPUs (T4, 16GB)
"""

print(qlora_explanation)

memory_comparison = pd.DataFrame([
    {"method": "Full fine-tuning (16-bit)", "7B_model_memory_gb": "~70GB+ (weights + gradients + optimizer states)"},
    {"method": "LoRA (16-bit base)", "7B_model_memory_gb": "~16GB (frozen base) + small LoRA overhead"},
    {"method": "QLoRA (4-bit base)", "7B_model_memory_gb": "~5-6GB (quantized base) + small LoRA overhead"},
])
memory_comparison


QLoRA = Quantized LoRA

LoRA already reduces TRAINABLE parameters dramatically.
QLoRA additionally reduces MEMORY usage of the FROZEN base model.

The problem QLoRA solves:
- A 7B parameter model in standard 16-bit precision needs ~14GB of GPU memory
  just to LOAD the model -- before any training even happens
- Most free/affordable GPUs (Colab free tier, consumer GPUs) have 8-16GB VRAM
- This makes fine-tuning even with LoRA inaccessible on modest hardware

QLoRA's fix:
1. Load the frozen base model in 4-bit precision instead of 16-bit
   -> Cuts memory for the frozen weights by ~4x
2. Apply LoRA adapters (in higher precision) on top of the quantized base
   -> Only the small LoRA matrices need higher precision for stable training
3. Use "double quantization" and paged optimizers for further memory savings

Result: a 7B model that needed ~14GB now fits in ~5-6GB
-> Makes fine-tuning possible on free Google Colab GPUs (T4, 16GB)



,method,7B_model_memory_gb
0,Full fine-tuning (16-bit),~70GB+ (weights + gradients + optimizer states)
1,LoRA (16-bit base),~16GB (frozen base) + small LoRA overhead
2,QLoRA (4-bit base),~5-6GB (quantized base) + small LoRA overhead


## 7. PEFT Library — Hands-On with Real Code

In [7]:
import sys
!{sys.executable} -m pip install peft transformers --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 8. PEFT — Configuring a LoRA Adapter

In [8]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification

# load a small model to demonstrate (we already have this from Day 48!)
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=5
)

# count parameters BEFORE LoRA
total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params:,}")

# configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                          # rank — the key hyperparameter from our math above
    lora_alpha=16,                 # scaling factor for LoRA updates
    lora_dropout=0.1,              # dropout for regularization
    target_modules=["q_lin", "v_lin"]  # which layers to apply LoRA to (attention query/value)
)

# wrap the model with LoRA
peft_model = get_peft_model(model, lora_config)

# count trainable parameters AFTER LoRA
peft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total model parameters: 66,957,317
trainable params: 741,893 || all params: 67,699,210 || trainable%: 1.0959


## 9. Inspecting What LoRA Actually Added

In [9]:
print("Modules LoRA was applied to:")
for name, module in peft_model.named_modules():
    if "lora" in name.lower() and ("lora_A" in name or "lora_B" in name):
        print(f"  {name}")

print(f"\nTotal trainable parameters: {sum(p.numel() for p in peft_model.parameters() if p.requires_grad):,}")
print(f"Total frozen parameters:    {sum(p.numel() for p in peft_model.parameters() if not p.requires_grad):,}")

Modules LoRA was applied to:
  base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A
  base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A.default
  base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B
  base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B.default
  base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_A
  base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_A.default
  base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_B
  base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_B.default
  base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A
  base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A.default
  base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B
  base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B.default
  base_model.model.distilbert.transformer.layer

## 10. Environment Setup Checklist for Day 61

In [10]:
import subprocess

checklist = """
DAY 61 PREP CHECKLIST — Fine-tuning a real LLM with QLoRA + Unsloth

Day 61 fine-tunes a 7-8B parameter model (Llama 3.1 8B or Phi-3).
This REQUIRES a GPU — CPU training would take days/weeks instead of minutes.

OPTIONS FOR GPU ACCESS:
[ ] Google Colab (free tier) — T4 GPU, 16GB VRAM, free, sufficient for QLoRA
    -> colab.research.google.com -> Runtime -> Change runtime type -> T4 GPU
[ ] Kaggle Notebooks (free tier) — P100 or T4 GPU, 30 hrs/week free
    -> kaggle.com/code -> New Notebook -> enable GPU in settings
[ ] Paid cloud GPU (if free tiers are insufficient) — RunPod, Lambda Labs, Vast.ai

WHAT TO DO BEFORE DAY 61:
[ ] Choose your GPU platform (Colab recommended — easiest, free, sufficient)
[ ] Create a Hugging Face account if you don't have one (you do — from Day 49!)
[ ] Generate a HF token with WRITE access (Settings -> Access Tokens)
      -> needed to push your fine-tuned model to the Hub at the end
[ ] Decide on a small custom dataset (100-1000 examples) for fine-tuning
      -> e.g. a Q&A format, instruction-following format, or style-specific text

WHAT WE'LL INSTALL ON DAY 61 (on the GPU platform):
  pip install unsloth
  -> Unsloth wraps PEFT + bitsandbytes (QLoRA) with custom CUDA kernels
  -> 2-5x faster training than standard PEFT, same final model quality
"""

print(checklist)


DAY 61 PREP CHECKLIST — Fine-tuning a real LLM with QLoRA + Unsloth

Day 61 fine-tunes a 7-8B parameter model (Llama 3.1 8B or Phi-3).
This REQUIRES a GPU — CPU training would take days/weeks instead of minutes.

OPTIONS FOR GPU ACCESS:
[ ] Google Colab (free tier) — T4 GPU, 16GB VRAM, free, sufficient for QLoRA
    -> colab.research.google.com -> Runtime -> Change runtime type -> T4 GPU
[ ] Kaggle Notebooks (free tier) — P100 or T4 GPU, 30 hrs/week free
    -> kaggle.com/code -> New Notebook -> enable GPU in settings
[ ] Paid cloud GPU (if free tiers are insufficient) — RunPod, Lambda Labs, Vast.ai

WHAT TO DO BEFORE DAY 61:
[ ] Choose your GPU platform (Colab recommended — easiest, free, sufficient)
[ ] Create a Hugging Face account if you don't have one (you do — from Day 49!)
[ ] Generate a HF token with WRITE access (Settings -> Access Tokens)
      -> needed to push your fine-tuned model to the Hub at the end
[ ] Decide on a small custom dataset (100-1000 examples) for fine-tuni

## 11. Summary — What We Learned Today

| Concept | Key Takeaway |
|---------|--------------|
| Decision framework | Try prompting first, RAG for knowledge gaps, fine-tune only for style/format/latency needs |
| Golden rule | 80% of "I need to fine-tune" problems are solved by better prompting |
| LoRA | Decomposes weight UPDATES into two small matrices (A, B) — freezes original weights |
| LoRA at scale | 99%+ parameter reduction on real LLM-sized layers (4096x4096) |
| QLoRA | Adds 4-bit quantization of the FROZEN base model — cuts memory ~4x |
| QLoRA impact | Makes 7B model fine-tuning possible on free 16GB Colab GPU (vs needing 70GB+) |
| PEFT library | Hugging Face's implementation — LoraConfig + get_peft_model() wraps any model |
| Our real result | DistilBERT: 741,893 trainable / 66,957,317 frozen = 1.1% trainable |
| target_modules | LoRA applied specifically to attention query/value layers, not the whole model |
| Day 61 prep | Need GPU access (Colab/Kaggle free tier), HF write token, and a small custom dataset |

**Key insight:**
LoRA and QLoRA didn't just make fine-tuning cheaper — they made it
ACCESSIBLE. What used to require a research lab's compute budget
(full fine-tuning) can now be done by an individual on a free Colab GPU.
This is precisely why fine-tuning skills are valuable in 2026 — the barrier
to entry collapsed, but most engineers still don't know how to do it properly.